# ЛР №4. Аналитическая витрина `hourly_rentals`: staging → mart

**Проект:** ITAD Labs — контур «Источник → raw → staging → mart → SQL-метрики».
**Датасет:** `hour_data` (UCI Bike Sharing Dataset, почасовые данные Capital Bikeshare, 2011–2012).
**Вход:** `s3://staging/hour_data/source_ingested_on=2026-09-16/staging.parquet` (результат ЛР №3).
**Выход:** `s3://mart/hour_data/source_ingested_on=2026-09-16/hourly_rentals.parquet`.

**Выбранный вопрос (основной из Project Brief):**
> Как погодные условия (температура, влажность, скорость ветра, тип погоды) и календарные факторы (сезон, месяц, день недели, рабочий день/праздник) влияют на суточное и почасовое количество аренд велосипедов в Capital Bikeshare?

Витрина дополнительно хранит разбивку на `casual` и `registered`, поэтому поддерживает и уточняющий вопрос №1: различается ли поведение зарегистрированных и случайных пользователей под влиянием тех же факторов.

**Метрики вопроса:** `cnt` — общее число аренд за час (шт.); доля зарегистрированных `registered / cnt × 100 %`; средняя часовая аренда по сезонам и погодным срезам.
**Срезы:** сезон, тип дня (рабочий / выходной или праздник), час суток, тип погоды, год.

**Смысл одной строки:** один час наблюдения системы велопроката в конкретную дату.
**Гранулярность:** час; она совпадает с гранулярностью staging, строки не агрегируются и не фильтруются.
**Ключ:** естественный составной `dteday + hr`; уникальность подтверждена в ЛР №3 (0 повторов).

Ноутбук переиспользует подключение DuckDB → SeaweedFS из ЛР №3, собирает витрину явным SQL-запросом, записывает Parquet в бакет `mart` и повторно читает его для проверок.

## Параметры запуска

Логическая дата `source_ingested_on` берётся из пути staging (дата загрузки исходного снимка), а не из даты выполнения ноутбука.

In [1]:
DATASET_SLUG = "hour_data"
INGESTED_ON = "2026-09-16"  # логическая дата загрузки исходного снимка (ЛР №2–3)

STAGING_S3_PATH = (
    f"s3://staging/{DATASET_SLUG}/source_ingested_on={INGESTED_ON}/staging.parquet"
)
MART_NAME = "hourly_rentals"
MART_PREFIX = f"s3://mart/{DATASET_SLUG}/source_ingested_on={INGESTED_ON}/"
MART_S3_PATH = MART_PREFIX + f"{MART_NAME}.parquet"

print("STAGING_S3_PATH =", STAGING_S3_PATH)
print("MART_S3_PATH    =", MART_S3_PATH)

STAGING_S3_PATH = s3://staging/hour_data/source_ingested_on=2026-09-16/staging.parquet
MART_S3_PATH    = s3://mart/hour_data/source_ingested_on=2026-09-16/hourly_rentals.parquet


## Подключение к SeaweedFS и чтение staging

Подключение то же, что в ЛР №3: DuckDB + расширение `httpfs`, параметры S3 из `.env` (по умолчанию `localhost:8333`, `s3admin` / `s3admin123`). `hive_partitioning=false` отключает автоматическое добавление служебного столбца `source_ingested_on` из пути.

In [2]:
import os

import duckdb
from IPython.display import display

try:
    from dotenv import load_dotenv, find_dotenv

    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    # python-dotenv не обязателен: значения по умолчанию совпадают с .env.example
    pass


def get_s3_endpoint() -> str:
    endpoint = os.getenv("S3_ENDPOINT", "localhost:8333")
    return endpoint.replace("http://", "").replace("https://", "").rstrip("/")


con = duckdb.connect()
try:
    con.execute("INSTALL httpfs")
except Exception:
    pass
con.execute("LOAD httpfs")
con.execute("SET enable_progress_bar = false")

con.execute(f"SET s3_endpoint = '{get_s3_endpoint()}'")
con.execute("SET s3_url_style = 'path'")
con.execute("SET s3_use_ssl = false")
con.execute("SET s3_region = 'us-east-1'")
con.execute(f"SET s3_access_key_id = '{os.getenv('S3_ACCESS_KEY', 's3admin')}'")
con.execute(f"SET s3_secret_access_key = '{os.getenv('S3_SECRET_KEY', 's3admin123')}'")

con.execute(
    f"CREATE OR REPLACE TEMP VIEW staging AS "
    f"SELECT * FROM read_parquet('{STAGING_S3_PATH}', hive_partitioning=false)"
)

display(con.execute("SELECT COUNT(*) AS staging_rows FROM staging").fetchdf())

,staging_rows
0,17379


## Шаг 1. Вход и аналитическая цель

Проверяем, что staging читается, содержит все необходимые для вопроса поля и что ограничения качества ЛР №3 не мешают расчёту метрик. Обязательные поля: дата и час наблюдения, календарные и погодные признаки, меры `casual`, `registered`, `cnt`, а также флаги качества.

In [3]:
REQUIRED_FIELDS = {
    "dteday", "hr", "season", "yr", "mnth", "holiday", "weekday", "workingday",
    "weathersit", "temp", "atemp", "hum", "windspeed", "casual", "registered", "cnt",
    "quality_status", "quality_flag", "quality_rule_ids",
}

staging_fields = set(con.execute("DESCRIBE staging").fetchdf()["column_name"])
missing = sorted(REQUIRED_FIELDS - staging_fields)
print("полей в staging:", len(staging_fields))
print("отсутствуют:", missing or "нет")
assert not missing, f"В staging нет обязательных полей: {missing}"

display(con.execute("""
SELECT COUNT(*) AS rows_cnt,
       MIN(dteday) AS min_date,
       MAX(dteday) AS max_date,
       COUNT(DISTINCT (dteday, hr)) AS distinct_keys,
       SUM(CASE WHEN quality_flag = 1 THEN 1 ELSE 0 END) AS flagged_rows
FROM staging
""").fetchdf())

полей в staging: 20
отсутствуют: нет


,rows_cnt,min_date,max_date,distinct_keys,flagged_rows
0,17379,2011-01-01,2012-12-31,17379,0.0


## Шаг 2. Гранулярность, ключ и меры

**Одна строка `hourly_rentals` представляет один час наблюдения системы велопроката в конкретную дату** — это тот же уровень детализации, что и в staging.

**Ключ:** `dteday + hr`. Естественный составной ключ: дата и час однозначно идентифицируют наблюдение; в ЛР №3 проверено, что повторов и пропусков по нему нет.

**Меры и правила агрегации:**

| Мера | Единица | Смысл | Суммирование |
|---|---|---|---|
| `cnt` | шт./час | всего аренд за час | суммируется по непересекающимся часам; средние — только с весами (по числу часов) |
| `casual` | шт./час | аренды случайных пользователей | то же |
| `registered` | шт./час | аренды зарегистрированных пользователей | то же |
| `registered_share_pct` | % | `registered / cnt × 100` | не суммируется и не усредняется без весов; это отношение, а не мера объёма |

`cnt = casual + registered` — согласованность проверена в ЛР №3 (правило R5, 0 нарушений).

**Время:** `dteday + hr` — время события (час наблюдения). `source_ingested_on=2026-09-16` — логическая дата загрузки снимка, она не заменяет время события и хранится в пути объекта.

**Структура:** одна широкая таблица. Для выбранного вопроса все описательные атрибуты (календарь, погода, сегменты) используются как срезы одной и той же меры, поэтому отдельные таблицы измерений не дают выигрыша: справочники малы и не переиспользуются другими витринами, а JOIN только усложнил бы проверки. Согласованная гранулярность «час» сохраняется.

**Фильтрация:** не применяется. Все 17 379 наблюдений staging остаются в витрине; решения о качестве ЛР №3 (quarantine, флаги) не пересматриваются.

## Шаг 3. Сборка витрины SQL-запросом

Явно перечисляем поля, переименовываем `yr` в `year_num` (2011/2012), добавляем метку времени наблюдения `observed_at`, человекочитаемые подписи срезов и долю зарегистрированных. Значения и флаги качества ЛР №3 переносятся без изменения.

In [4]:
con.execute("DROP VIEW IF EXISTS mart_hourly_rentals")
con.execute("""
CREATE TEMP VIEW mart_hourly_rentals AS
SELECT
    CAST(dteday AS DATE)                                   AS dteday,
    CAST(hr AS INTEGER)                                    AS hr,
    CAST(dteday AS TIMESTAMP) + to_hours(CAST(hr AS BIGINT)) AS observed_at,

    CAST(season AS INTEGER)                                AS season,
    CASE CAST(season AS INTEGER)
        WHEN 1 THEN 'Зима' WHEN 2 THEN 'Весна'
        WHEN 3 THEN 'Лето' WHEN 4 THEN 'Осень'
    END                                                    AS season_label,

    CAST(yr AS INTEGER)                                    AS yr,
    2011 + CAST(yr AS INTEGER)                             AS year_num,
    CAST(mnth AS INTEGER)                                  AS mnth,
    CAST(holiday AS INTEGER)                               AS holiday,

    CAST(weekday AS INTEGER)                               AS weekday,
    CASE CAST(weekday AS INTEGER)
        WHEN 0 THEN 'Вс' WHEN 1 THEN 'Пн' WHEN 2 THEN 'Вт' WHEN 3 THEN 'Ср'
        WHEN 4 THEN 'Чт' WHEN 5 THEN 'Пт' WHEN 6 THEN 'Сб'
    END                                                    AS weekday_label,
    CAST(workingday AS INTEGER)                            AS workingday,
    CASE WHEN CAST(workingday AS INTEGER) = 1
         THEN 'Рабочий день' ELSE 'Выходной или праздник'
    END                                                    AS day_type,

    CAST(weathersit AS INTEGER)                            AS weathersit,
    CASE CAST(weathersit AS INTEGER)
        WHEN 1 THEN 'Ясно'
        WHEN 2 THEN 'Туман/облачно'
        WHEN 3 THEN 'Лёгкий дождь/снег'
        WHEN 4 THEN 'Сильный дождь/снег/туман'
    END                                                    AS weather_label,

    CAST(temp AS DOUBLE)                                   AS temp,
    CAST(atemp AS DOUBLE)                                  AS atemp,
    CAST(hum AS DOUBLE)                                    AS hum,
    CAST(windspeed AS DOUBLE)                              AS windspeed,

    CAST(casual AS BIGINT)                                 AS casual,
    CAST(registered AS BIGINT)                             AS registered,
    CAST(cnt AS BIGINT)                                    AS cnt,
    ROUND(100.0 * CAST(registered AS DOUBLE)
          / NULLIF(CAST(cnt AS BIGINT), 0), 2)             AS registered_share_pct,

    quality_status,
    CAST(quality_flag AS INTEGER)                          AS quality_flag,
    quality_rule_ids
FROM staging
""")

display(con.execute("SELECT COUNT(*) AS mart_rows FROM mart_hourly_rentals").fetchdf())
display(con.execute("SELECT * FROM mart_hourly_rentals LIMIT 3").fetchdf())

,mart_rows
0,17379


,dteday,hr,observed_at,season,season_label,yr,year_num,mnth,holiday,weekday,...,atemp,hum,windspeed,casual,registered,cnt,registered_share_pct,quality_status,quality_flag,quality_rule_ids
0,2011-01-01,0,2011-01-01 00:00:00,1,Зима,0,2011,1,0,6,...,0.2879,0.81,0.0,3,13,16,81.25,clean,0,
1,2011-01-01,1,2011-01-01 01:00:00,1,Зима,0,2011,1,0,6,...,0.2727,0.80,0.0,8,32,40,80.00,clean,0,
2,2011-01-01,2,2011-01-01 02:00:00,1,Зима,0,2011,1,0,6,...,0.2727,0.80,0.0,5,27,32,84.38,clean,0,


## Шаг 4. Запись витрины в S3

Материализуем view в Parquet в бакете `mart`. Повторное выполнение перезаписывает объект предыдущего запуска. Фильтрация не выполняется, поэтому число строк до и после записи должно совпасть.

In [5]:
staging_count = con.execute("SELECT COUNT(*) FROM staging").fetchone()[0]
mart_count = con.execute("SELECT COUNT(*) FROM mart_hourly_rentals").fetchone()[0]

con.execute(
    f"COPY (SELECT * FROM mart_hourly_rentals) TO '{MART_S3_PATH}' (FORMAT PARQUET);"
)

excluded_count = staging_count - mart_count
print("staging rows =", staging_count)
print("mart rows    =", mart_count)
print("excluded     =", excluded_count)
assert mart_count == staging_count, "Витрина изменила число строк без обоснованной фильтрации"
print("записано:", MART_S3_PATH)

staging rows = 17379
mart rows    = 17379
excluded     = 0
записано: s3://mart/hour_data/source_ingested_on=2026-09-16/hourly_rentals.parquet


## Шаг 5. Проверки записанного результата

Читаем объект из `mart` заново и проверяем количество строк и схему, ключ, меры, диапазоны, перенос флагов качества и пригодность для выбранного вопроса.

In [6]:
display(con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{MART_S3_PATH}', hive_partitioning=false)"
).fetchdf())

display(con.execute("""
SELECT COUNT(*) AS mart_rows,
       MIN(observed_at) AS min_observed_at,
       MAX(observed_at) AS max_observed_at
FROM read_parquet(?, hive_partitioning=false)
""", [MART_S3_PATH]).fetchdf())

mart_s3_count = con.execute(
    "SELECT COUNT(*) FROM read_parquet(?, hive_partitioning=false)", [MART_S3_PATH]
).fetchone()[0]
print("staging =", staging_count, "| mart_s3 =", mart_s3_count, "| excluded =", excluded_count)
assert staging_count == mart_s3_count + excluded_count

,column_name,column_type,null,key,default,extra
0,dteday,DATE,YES,None,None,None
1,hr,INTEGER,YES,None,None,None
2,observed_at,TIMESTAMP,YES,None,None,None
3,season,INTEGER,YES,None,None,None
4,season_label,VARCHAR,YES,None,None,None
5,yr,INTEGER,YES,None,None,None
6,year_num,INTEGER,YES,None,None,None
7,mnth,INTEGER,YES,None,None,None
8,holiday,INTEGER,YES,None,None,None
9,weekday,INTEGER,YES,None,None,None


,mart_rows,min_observed_at,max_observed_at
0,17379,2011-01-01,2012-12-31 23:00:00


staging = 17379 | mart_s3 = 17379 | excluded = 0


### Ключ и обязательные поля

Проверяем пропуски в компонентах ключа и повторы пар `dteday + hr`, а также заполненность полей, без которых теряется смысл строки.

In [7]:
display(con.execute("""
SELECT
    SUM(CASE WHEN dteday IS NULL THEN 1 ELSE 0 END) AS null_dteday,
    SUM(CASE WHEN hr IS NULL THEN 1 ELSE 0 END)     AS null_hr,
    SUM(CASE WHEN observed_at IS NULL THEN 1 ELSE 0 END) AS null_observed_at,
    SUM(CASE WHEN season IS NULL OR weathersit IS NULL THEN 1 ELSE 0 END) AS null_slice_fields
FROM read_parquet(?, hive_partitioning=false)
""", [MART_S3_PATH]).fetchdf())

display(con.execute("""
SELECT dteday, hr, COUNT(*) AS rows_per_key
FROM read_parquet(?, hive_partitioning=false)
GROUP BY dteday, hr
HAVING COUNT(*) > 1
""", [MART_S3_PATH]).fetchdf())

,null_dteday,null_hr,null_observed_at,null_slice_fields
0,0.0,0.0,0.0,0.0


,dteday,hr,rows_per_key


### Меры и вычисленные поля

Проверяем отсутствие NULL в мерах, минимальные и максимальные значения, диапазоны `hr` (0–23), `weathersit` (1–4) и доли зарегистрированных (0–100 %).

In [8]:
display(con.execute("""
SELECT
    SUM(CASE WHEN cnt IS NULL OR casual IS NULL OR registered IS NULL THEN 1 ELSE 0 END) AS null_measures,
    MIN(cnt) AS min_cnt, MAX(cnt) AS max_cnt,
    MIN(casual) AS min_casual, MAX(casual) AS max_casual,
    MIN(registered) AS min_registered, MAX(registered) AS max_registered,
    SUM(CASE WHEN hr < 0 OR hr > 23 THEN 1 ELSE 0 END) AS hr_out_of_range,
    SUM(CASE WHEN weathersit NOT BETWEEN 1 AND 4 THEN 1 ELSE 0 END) AS weathersit_out_of_range,
    SUM(CASE WHEN registered_share_pct < 0 OR registered_share_pct > 100 THEN 1 ELSE 0 END) AS share_out_of_range,
    SUM(CASE WHEN cnt <> casual + registered THEN 1 ELSE 0 END) AS consistency_violations
FROM read_parquet(?, hive_partitioning=false)
""", [MART_S3_PATH]).fetchdf())

,null_measures,min_cnt,max_cnt,min_casual,max_casual,min_registered,max_registered,hr_out_of_range,weathersit_out_of_range,share_out_of_range,consistency_violations
0,0.0,1,977,0,367,0,886,0.0,0.0,0.0,0.0


### Флаги качества ЛР №3

Сверяем количество помеченных строк со staging и проверяем, что значения `quality_status`, `quality_flag` и `quality_rule_ids` перенесены без изменений. В данном снимке все строки `clean`, `quality_flag = 0`.

In [9]:
display(con.execute("""
SELECT quality_status, quality_flag, COUNT(*) AS rows_cnt
FROM read_parquet(?, hive_partitioning=false)
GROUP BY quality_status, quality_flag ORDER BY quality_flag
""", [MART_S3_PATH]).fetchdf())

display(con.execute("""
SELECT COUNT(*) AS flag_mismatch_rows
FROM read_parquet(?, hive_partitioning=false) m
JOIN staging s USING (dteday, hr)
WHERE m.quality_flag IS DISTINCT FROM s.quality_flag
   OR m.quality_status IS DISTINCT FROM s.quality_status
   OR m.quality_rule_ids IS DISTINCT FROM s.quality_rule_ids
""", [MART_S3_PATH]).fetchdf())

,quality_status,quality_flag,rows_cnt
0,clean,0,17379


,flag_mismatch_rows
0,0


### Пригодность для выбранного вопроса

Короткий срез: средняя часовая аренда и средняя доля зарегистрированных по типам погоды. Запрос подтверждает, что витрина поддерживает сравнение спроса по погодному срезу при сохранённой гранулярности «час».

In [10]:
display(con.execute("""
SELECT weathersit, weather_label,
       COUNT(*) AS hours_cnt,
       ROUND(AVG(cnt), 1) AS avg_cnt_per_hour,
       ROUND(AVG(registered_share_pct), 1) AS avg_registered_share_pct
FROM read_parquet(?, hive_partitioning=false)
GROUP BY weathersit, weather_label
ORDER BY weathersit
""", [MART_S3_PATH]).fetchdf())

,weathersit,weather_label,hours_cnt,avg_cnt_per_hour,avg_registered_share_pct
0,1,Ясно,11413,204.9,81.7
1,2,Туман/облачно,4544,175.2,84.0
2,3,Лёгкий дождь/снег,1419,111.6,87.3
3,4,Сильный дождь/снег/туман,3,74.3,96.4


## Итоги и ограничения

- Витрина `hourly_rentals` записана в `s3://mart/hour_data/source_ingested_on=2026-09-16/hourly_rentals.parquet` и повторно прочитана из S3.
- Гранулярность «один час наблюдения» и число строк сохранены: 17 379 = 17 379 + 0 исключённых. Фильтрация не применялась, решения о качестве ЛР №3 не пересматривались.
- Ключ `dteday + hr` без пропусков и повторов; меры заполнены, `cnt = casual + registered`; диапазоны часов и типов погоды корректны.
- Флаги качества ЛР №3 перенесены; расхождений со staging нет.
- JOIN не используются, поэтому проверки кардинальности и ненайденных соответствий не требуются.

**Ограничения.** Данные агрегированы по всей системе (нет станций), охватывают только 2011–2012 годы, погодные показатели нормализованы; витрина не содержит разбивки по станциям и не является историей изменений — это статичный снимок наблюдений. Доля `registered_share_pct` — отношение, её нельзя суммировать или усреднять без весов. Суточные итоги в витрину не включены: они получаются агрегацией часовых строк по `dteday` на следующем шаге контура (SQL-метрики).